In [ ]:

# llpsmd_Jun_nuc
import MDAnalysis as mda # 2.3.0, by "mda.version.__version__"
# from MDAnalysis.tests.datafiles import PSF, DCD
from MDAnalysis.analysis import contacts, distances
from MDAnalysis import transformations as trans
from scipy import sparse
import nglview as nv
import os
import fnmatch as fm
import numpy as np
import math
import pandas as pd
import pickle
import matplotlib.pyplot as plt
import parmed
import numpy as np
from scipy.optimize import curve_fit
import matplotlib.pyplot as plt


In [ ]:

# define functions
from scipy import optimize
from scipy import sparse

def timeCorr_bool(mats, N, m):
    # mats is a list of bool matrix, which express the bindings
    # N= len(mats)
    # m: time region of correlation, should < N/2
    corr_m= np.zeros(m)
    #cnt= 1
    for i in range(m):
        # matCorr= []
        matCorr= np.zeros(N-i)
        for k in range(N-i):
            matCorr[k]= (mats[k] & mats[k+i]).sum()
            # matCorr.append( (mats[k] * mats[k+i]).sum() )
        corr_m[i]= np.mean(matCorr)
        
        '''if i>= ((cnt*m)//25)-1:
            print(f"{cnt*25} %")
            cnt+=1'''
        
    corrN_m= corr_m/corr_m[0]
    return corrN_m, corr_m

def timeCorr_boolCsr(mats, N, m):
    # mats is a list of bool csr_matrix, which express the bindings
    # N= len(mats)
    # m: time region of correlation, should < N/2
    corr_m= np.zeros(m)
    # cnt= 1
    for i in range(m):
        # matCorr= []
        matCorr= np.zeros(N-i)
        for k in range(N-i):
            matCorr[k]= (mats[k].multiply(mats[k+i])).sum()
        corr_m[i]= np.mean(matCorr)
        
        '''if i>= ((cnt*m)//10)-1:
            print(f"{cnt*10} %")
            cnt+=1'''
        
    corrN_m= corr_m/corr_m[0]
    return corrN_m, corr_m

def exp2(t, a, tau1, c):
    return a * np.exp(-t / tau1) + (1 - a) * np.exp(-t / (c*tau1))

def fit_corr(Ct, t, init):
    # fit normalized correlation ( C(0)=1 )
    n= len(t)
    popt2, pcov2 = optimize.curve_fit(
            exp2, t, Ct, sigma= (t+t[1])**(1/3.0),
            bounds=([0,0,1],[1,10*t[-1],math.inf]), p0=init, maxfev=1000000) # sigma= t**0.5,
    
    Ct_fit= np.zeros(n)
    for i in range(n):
        Ct_fit[i]= exp2(t[i], *popt2)
    return Ct_fit, popt2    

def calc_contact(grp1, grp2, d, dims):
    # calculate contact matrix
    dist= contacts.distance_array(grp1.positions, grp2.positions, box=dims)
    contMat_12= contacts.contact_matrix(dist, radius=d)
    return contMat_12


def timeCorr_boolCsr_fromStart(mats, N, m):
    # mats is a list of bool csr_matrix, which express the bindings
    # N= len(mats)
    # m: time region of correlation, should < N/2
    corr_m = np.zeros(m)
    
    for i in range(m):
        corr_m[i] = (mats[0].multiply(mats[i])).sum()
    
    corrN_m = corr_m / corr_m[0]
    return corrN_m, corr_m


def visul5(unive,s1,s2):
    view=nv.show_mdanalysis(unive)
    view.clear_representations()
    view.add_representation('ball+stick',sele=f"(nucleic) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='red')  
    
    view.add_representation('ball+stick',sele=f"(protein) and ( {s1})",
                            opacity=1.5,aspectRatio=2,color='yellow')  
    

    view.add_representation('ball+stick',sele=f"(nucleic or protein ) and ({s2})",
                            opacity=1.5,
                            aspectRatio=2,
                            color='green')
    
    

    view.add_unitcell(color='white')
    return view


In [ ]:
stru = parmed.load_file('../gxg_comps/3_res/equal/output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/3_res/equal/traj_2-nw.dcd')

In [ ]:
stru = parmed.load_file('../gxg_comps/3_res/aromatic/output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/3_res/aromatic/traj_2-nw.dcd')

In [ ]:
stru = parmed.load_file('../gxg_comps/3_res/nonpolar//output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/3_res/nonpolar//traj_2-nw.dcd')

In [ ]:
stru = parmed.load_file('../gxg_comps/3_res//output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/3_res/n/traj_2-nw.dcd')

In [ ]:
u.trajectory

In [ ]:

a= u.atoms
# --- wrap trajectory (do only once if needed)
trans_wrap = trans.wrap(a)
trans_unwrap= trans.unwrap(a)
workflow= [trans.wrap(a), trans.unwrap(a)] # unwrap() to make chain whole...
u.trajectory.add_transformations(*workflow) 


In [ ]:
# ── Parameters ────────────────────────────────────────────────────────────────
label = 'equal'
clrs  = ['#E01010', '#1010E0', '#10A010']
clrs2 = ['#E09010', '#1090E0', '#10C040']
d_cont     = 8       # Å
#nM         = 500
chain_size = 3       # residues per RNA chain
results = {}
star = 2000
nM = int(star//2.12)
dt = 2 ## (in ps)
idx =1

# -- Atom selections ------------------------------------------------------
idp_1 = u.select_atoms("name CA")[::20]   # one per residue  (N_1,)
idp_2 = u.select_atoms("name CA")        # all O4'          (N_2,)

# -- Intra-chain mask (built once, chains by residue index) ---------------
resids_1 = idp_1.resids                              # (N_1,)
resids_2 = idp_2.resids                              # (N_2,)
chain_1  = (resids_1 - resids_1.min()) // chain_size # (N_1,)
chain_2  = (resids_2 - resids_2.min()) // chain_size # (N_2,)
intra_mask = (chain_1[:, None] == chain_2[None, :])  # (N_1, N_2) bool
print(f"  idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
      f"chains: {chain_1.max()+1}")

# -- Contact matrix accumulation ------------------------------------------


contMt = []
total_frames  = len(u.trajectory)
frame_indices = list(range(max(0, total_frames - nM), total_frames))

for _ in u.trajectory[-star:]:
    contMat_rna = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
    contMat_rna[intra_mask] = False          # zero intra-chain contacts
    contMt.append(sparse.csr_matrix(contMat_rna))
print("  Contact matrix done.")

# -- Time axis ------------------------------------------------------------
tM = dt * np.arange(nM)   # [ns]

# -- Time correlation -----------------------------------------------------

##corrN, corr = timeCorr_boolCsr(contMt,len(contMt), nM)

corrN, corr = timeCorr_boolCsr_fromStart(contMt,len(contMt), nM)
print("  Time correlation done.")

# -- Exponential fitting --------------------------------------------------
corrFit, para = fit_corr(corrN, tM, [0, tM[-1] / 4, 10])
t1    = para[1]
t2    = para[1] * para[2]
t_eff = para[0] * para[1] + (1 - para[0]) * (para[1] * para[2])
print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")

# -- Store results --------------------------------------------------------
results[label] = {
    'tM':      tM,
    'corrN':   corrN,
    'corr':    corr,
    'corrFit': corrFit,
    'para':    para,
    't1':      t1,
    't2':      t2,
    't_eff':   t_eff,
}

# -- Individual plot ------------------------------------------------------
plt.rcParams.update({"font.size": 18})
plt.figure()
plt.plot(tM, corrN,   ".",  color=clrs[idx],  label="data")
plt.plot(tM, corrFit, "--", color=clrs2[idx], label=f"fit  τ_eff={t_eff:.2f} ns")
plt.title(f"Protein  T={label} K")
plt.xlabel("t [ns]")
plt.ylabel("C(t)")
plt.legend(frameon=False, fontsize=14)
plt.tight_layout()
plt.savefig(f"rna_u_corr_T{label}.png", dpi=150)
plt.show()

In [ ]:
u.select_atoms('resid 0 1 2 3 4')

In [ ]:
### Correaltion only of condensate

In [ ]:
import parmed
from sklearn.cluster import DBSCAN
from MDAnalysis.analysis import contacts
from MDAnalysis.analysis import distances
from collections import Counter
import nglview as nv
import MDAnalysis as mda
import numpy as np
from MDAnalysis.transformations import wrap, center_in_box


### For RNA G and A, it is 13
### For RNA U and C, it is 11.5

def clustering(u, frame,eps=13):

        ### Function that is nested in all of the analysis methods in this file ###

        ca_sel = u.select_atoms("name C1' or name CA")

        cas = [atom.index for atom in ca_sel.atoms]

        to_cluster = [atom for i, atom in enumerate(u.trajectory[frame.frame].positions)if i in cas]

        n_ca = len(ca_sel)

        self_distances = distances.self_distance_array(ca_sel.positions,
                                                       box=frame.dimensions)

        sq_dist_arr = np.zeros((n_ca, n_ca))
        triu = np.triu_indices_from(sq_dist_arr, k=1)

        sq_dist_arr[triu] = self_distances
        sq_dist_arr.T[triu] = self_distances
  
        model = DBSCAN(eps=eps, min_samples=1, metric='precomputed')
    
        labels = model.fit_predict(sq_dist_arr)
        b = Counter(labels)

        clustered_list = [atom for j, atom in enumerate(to_cluster)
                          for i, item in enumerate(labels)
                          if item == b.most_common()[0][0]
                          and j == i]

        centroid = np.mean(clustered_list, axis=0)

        ind_list = [cas[index] for index, j in enumerate(to_cluster)
                    for i in clustered_list
                    if j[0]==i[0] and j[1]==i[1] and j[2]==i[2]]

        index_list = [x+1 for x in ind_list]

        res_list = [u.atoms[i].resindex for i in index_list]

        return centroid, to_cluster, clustered_list, res_list


In [ ]:
stru = parmed.load_file('../gxg_comps//3_res/equal/output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/3_res/equal//traj_2-nw.dcd')

In [ ]:
stru = parmed.load_file('../gxg_comps/3_res/aromatic/output.top')

# or explicitly by atom names:
stru.strip('@OW,HW1,HW2')
u = mda.Universe(stru,
                 '../gxg_comps/3_res/aromatic/traj_2-nw.dcd')

In [ ]:
a= u.atoms
# --- wrap trajectory (do only once if needed)
trans_wrap = trans.wrap(a)
trans_unwrap= trans.unwrap(a)
workflow= [trans.wrap(a), trans.unwrap(a)] # unwrap() to make chain whole...
u.trajectory.add_transformations(*workflow) 

In [ ]:
start = -6000

u=univ

ts = u.trajectory[start]  
centroid, to_cluster, clustered_list, res_list = clustering(u, ts,eps=9)

selection = 'resindex '
for item in res_list:
        selection += str(item)+ ' or resindex '
selection = selection[0:-13]

res_list = np.array(res_list)

selection_2 = ' '
for item in res_list:
        selection_2 += str(item)+ ' '       

sel = np.unique(univ.select_atoms(f"(nucleic or protein) and not ({selection})").resids)

selection_3 = ' '
for item in sel:
        selection_3 += str(item)+ ' '

In [ ]:

# ── Parameters ────────────────────────────────────────────────────────────────
label = 'equal'
clrs  = ['#E01010', '#1010E0', '#10A010']
clrs2 = ['#E09010', '#1090E0', '#10C040']
d_cont     = 6       # Å
#nM         = 500
chain_size = 3       # residues per RNA chain
results = {}
star = 6000
nM = int(star//2.12)
dt = 2 ## (in ps)
idx =1

# -- Atom selections ------------------------------------------------------
idp_1 = u.select_atoms(f"name CA and (resid {selection_2})")[::2]   # one per residue  (N_1,)
idp_2 = u.select_atoms("name CA")        # all O4'          (N_2,)

# -- Intra-chain mask (built once, chains by residue index) ---------------
resids_1 = idp_1.resids                              # (N_1,)
resids_2 = idp_2.resids                              # (N_2,)
chain_1  = (resids_1) // chain_size # (N_1,)
chain_2  = (resids_2) // chain_size # (N_2,)
intra_mask = (chain_1[:, None] == chain_2[None, :])  # (N_1, N_2) bool


print(f"  idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
      f"chains: {chain_1.max()+1}")

# -- Contact matrix accumulation ------------------------------------------


contMt = []
total_frames  = len(u.trajectory)
frame_indices = list(range(max(0, total_frames - nM), total_frames))

for _ in u.trajectory[-star:]:
    contMat_rna = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
    contMat_rna[intra_mask] = False          # zero intra-chain contacts
    contMt.append(sparse.csr_matrix(contMat_rna))
print("  Contact matrix done.")

# -- Time axis ------------------------------------------------------------
tM = dt * np.arange(nM)   # [ns]

# -- Time correlation -----------------------------------------------------

##corrN, corr = timeCorr_boolCsr(contMt,len(contMt), nM)

corrN, corr = timeCorr_boolCsr_fromStart(contMt,len(contMt), nM)
print("  Time correlation done.")

# -- Exponential fitting --------------------------------------------------
corrFit, para = fit_corr(corrN, tM, [0, tM[-1] / 4, 10])
t1    = para[1]
t2    = para[1] * para[2]
t_eff = para[0] * para[1] + (1 - para[0]) * (para[1] * para[2])
print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")

# -- Store results --------------------------------------------------------
results[label] = {
    'tM':      tM,
    'corrN':   corrN,
    'corr':    corr,
    'corrFit': corrFit,
    'para':    para,
    't1':      t1,
    't2':      t2,
    't_eff':   t_eff,
}

# -- Individual plot ------------------------------------------------------
plt.rcParams.update({"font.size": 18})
plt.figure()
plt.plot(tM, corrN,   ".",  color=clrs[idx],  label="data")
plt.plot(tM, corrFit, "--", color=clrs2[idx], label=f"fit  τ_eff={t_eff:.2f} ns")
plt.title(f"Protein  T={label} K")
plt.xlabel("t [ns]")
plt.ylabel("C(t)")
plt.legend(frameon=False, fontsize=14)
plt.tight_layout()
plt.savefig(f"rna_u_corr_T{label}.png", dpi=150)
plt.show()


### Apply on all systems 

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Main analysis loop
# ══════════════════════════════════════════════════════════════════════════════
 
systems    = ['equal', 'aromatic', 'nonpolar', 'polar', 'charged']
clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']
 
# ── Parameters ────────────────────────────────────────────────────────────────
d_cont     = 8        # Å  contact cutoff
chain_size = 3        # residues per RNA chain
star       = 2000     # frames to analyse
m_corr     = 2000   # correlation window  (~2830)
dt         = 2e-3     # ps per frame → ns  (2 ps = 0.002 ns)
eps_clust  = 9        # DBSCAN eps for clustering
 
save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)
 
results = {}
 
for idx, label in enumerate(systems):
    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")
 
    # ── Load universe ─────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/3_res/{label}/output.top')
    stru.strip('@OW,HW1,HW2')
    u = mda.Universe(stru, f'../gxg_comps/3_res/{label}/traj_2-nw.dcd')
 
    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)
 
    # ── Clustering to find dense cluster ──────────────────────────────────────
    ts = u.trajectory[-star]
    centroid, to_cluster, clustered_list, res_list = clustering(u, ts, eps=eps_clust)
    
    
    selection = 'resindex '
    for item in res_list:
            selection += str(item)+ ' or resindex '
    selection = selection[0:-13]

    res_list = np.array(res_list)

    selection_2 = ' '
    for item in res_list:
            selection_2 += str(item)+ ' '       

    sel = np.unique(univ.select_atoms(f"(nucleic or protein) and not ({selection})").resids)

    selection_3 = ' '
    for item in sel:
            selection_3 += str(item)+ ' '



    # ── FIXED: both idp_1 and idp_2 are dilute-phase CA only ─────────────────
    idp_1 = u.select_atoms(f"name CA and (resid {selection_2})")[::2]
    idp_2 = u.select_atoms(f"(name CA)")
 
    # ── Intra-chain mask ───────────────────────────────────────────────────────
    resids_1   = idp_1.resids
    resids_2   = idp_2.resids
    chain_1    = resids_1 // chain_size
    chain_2    = resids_2 // chain_size
    intra_mask = (chain_1[:, None] == chain_2[None, :])
 
    print(f"  idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
          f"chains: {chain_1.max() + 1}  |  dilute residues: {len(sel_dilute)}")
 
    # ── Contact matrix accumulation ───────────────────────────────────────────
    contMt = []
    for _ in u.trajectory[-star:]:
        contMat = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
        contMat[intra_mask] = False
        contMt.append(sparse.csr_matrix(contMat))
    print(f"  Contact matrix done. ({len(contMt)} frames)")
 
    # ── Time axis (in ns) ─────────────────────────────────────────────────────
    tM = dt * np.arange(m_corr)   # ns
 
    # ── Time correlation ───────────────────────────────────────────────────────
    corrN, corr = timeCorr_boolCsr_fromStart(contMt, len(contMt), m_corr)
    #print(f"  Time correlation done.  baseline (frozen fraction) = {baseline/corr[0]:.3f}")
 
    # ── Exponential fitting ────────────────────────────────────────────────────
    # FIXED: initial guess a=0.5, not 0
    corrFit, para = fit_corr(corrN, tM, [0.5, tM[-1] / 4, 10])
    t1    = para[1]
    t2    = para[1] * para[2]
    t_eff = para[0] * para[1] + (1 - para[0]) * t2
    print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")
 
    # ── Store results ──────────────────────────────────────────────────────────
    results[label] = {
        'tM':      tM,
        'corrN':   corrN,
        'corr':    corr,
        #'baseline': baseline,
        'corrFit': corrFit,
        'para':    para,
        't1':      t1,
        't2':      t2,
        't_eff':   t_eff,
    }

#### ── Save all results ───────────────────────────────────────────────────────────
save_path = os.path.join(save_dir, 'corr_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Results saved to {save_path}")



In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Load and plot
# ══════════════════════════════════════════════════════════════════════════════
 
with open(save_path, 'rb') as f:
    results = pickle.load(f)
print(f"  Results loaded from {save_path}")
 
plt.rcParams.update({"font.size": 18})
 
# ── Individual plots ───────────────────────────────────────────────────────────
for idx, label in enumerate(systems):
    r = results[label]
    plt.figure()
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],
             label='data')
    plt.plot(r['tM'], r['corrFit'], '--', color=clrs2[idx],
             label=f"fit  τ_eff={r['t_eff']:.2f} ns")
    plt.title(f"Protein contacts — {label}")
    plt.xlabel("t [ns]")
    plt.ylabel("C(t)")
    plt.legend(frameon=False, fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"corr_{label}.png"), dpi=150)
    plt.show()
    print(f"  Saved plot: corr_{label}.png")
 
# ── Combined comparison plot ───────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
for idx, label in enumerate(systems):
    r = results[label]
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],  alpha=0.4)
    plt.plot(r['tM'], r['corrFit'], '-', color=clrs[idx],
             label=f"{label}  τ_eff={r['t_eff']:.1f} ns")
plt.xlabel("t [ns]")
plt.ylabel("C(t)")
plt.title("Protein contact correlation — all systems")
plt.legend(frameon=False, fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "corr_all_systems.png"), dpi=150)
plt.show()
print("  Saved combined plot: corr_all_systems.png")
 
# ── τ_eff summary bar chart ────────────────────────────────────────────────────
t_effs = [results[s]['t_eff'] for s in systems]
plt.figure(figsize=(8, 5))
plt.bar(systems, t_effs, color=clrs)
plt.ylabel("τ_eff [ns]")
plt.title("Effective binding lifetime")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "tau_eff_comparison.png"), dpi=150)
plt.show()
print("  Saved τ_eff bar chart: tau_eff_comparison.png")

In [ ]:
import parmed
from sklearn.cluster import DBSCAN
from MDAnalysis.analysis import contacts
from MDAnalysis.analysis import distances
from collections import Counter
import nglview as nv
import MDAnalysis as mda
import MDAnalysis.transformations as trans
import numpy as np
from scipy import optimize, sparse
import math
import matplotlib.pyplot as plt
import pickle
import os

# ══════════════════════════════════════════════════════════════════════════════
#  Helper functions (unchanged from your originals)
# ══════════════════════════════════════════════════════════════════════════════

def clustering(u, frame, eps=13):
    ca_sel = u.select_atoms("name C1' or name CA")
    cas = [atom.index for atom in ca_sel.atoms]
    to_cluster = [atom for i, atom in enumerate(u.trajectory[frame.frame].positions)
                  if i in cas]
    n_ca = len(ca_sel)
    self_distances = distances.self_distance_array(ca_sel.positions,
                                                   box=frame.dimensions)
    sq_dist_arr = np.zeros((n_ca, n_ca))
    triu = np.triu_indices_from(sq_dist_arr, k=1)
    sq_dist_arr[triu] = self_distances
    sq_dist_arr.T[triu] = self_distances

    model  = DBSCAN(eps=eps, min_samples=1, metric='precomputed')
    labels = model.fit_predict(sq_dist_arr)
    b      = Counter(labels)

    clustered_list = [atom for j, atom in enumerate(to_cluster)
                      for i, item in enumerate(labels)
                      if item == b.most_common()[0][0] and j == i]
    centroid = np.mean(clustered_list, axis=0)
    ind_list = [cas[index] for index, j in enumerate(to_cluster)
                for i in clustered_list
                if j[0] == i[0] and j[1] == i[1] and j[2] == i[2]]
    index_list = [x + 1 for x in ind_list]
    res_list   = [u.atoms[i].resindex for i in index_list]
    return centroid, to_cluster, clustered_list, res_list


def timeCorr_boolCsr_fromStart(mats, N, m):
    corr_m = np.zeros(m)
    for i in range(m):
        corr_m[i] = (mats[0].multiply(mats[i])).sum()
    # subtract plateau (last 20 %) and renormalise
    baseline  = np.mean(corr_m[int(0.8 * m):])
    corr_sub  = corr_m - baseline
    corrN_m   = corr_sub / corr_sub[0]
    return corrN_m, corr_m, baseline


def exp2(t, a, tau1, c):
    return a * np.exp(-t / tau1) + (1 - a) * np.exp(-t / (c * tau1))


def fit_corr(Ct, t, init):
    n = len(t)
    popt2, pcov2 = optimize.curve_fit(
        exp2, t, Ct, sigma=(t + t[1]) ** (1 / 3.0),
        bounds=([0, 0, 1], [1, 10 * t[-1], math.inf]),
        p0=init, maxfev=1000000)
    Ct_fit = np.array([exp2(t[i], *popt2) for i in range(n)])
    return Ct_fit, popt2


def calc_contact(sel1, sel2, cutoff, box):
    """Return bool matrix (N1 x N2): True where distance < cutoff."""
    dist_mat = distances.distance_array(sel1.positions, sel2.positions, box=box)
    return dist_mat < cutoff


# ══════════════════════════════════════════════════════════════════════════════
#  Main analysis loop
# ══════════════════════════════════════════════════════════════════════════════

systems    = ['equal', 'aromatic', 'nonpolar', 'polar', 'charged']
clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']

# ── Parameters ────────────────────────────────────────────────────────────────
d_cont     = 6        # Å  contact cutoff
chain_size = 3        # residues per RNA chain
star       = 6000     # frames to analyse
m_corr     = int(star // 2.12)   # correlation window  (~2830)
dt         = 2e-3     # ps per frame → ns  (2 ps = 0.002 ns)
eps_clust  = 9        # DBSCAN eps for clustering

save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)

results = {}

for idx, label in enumerate(systems):
    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")

    # ── Load universe ─────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/3_res/{label}/output.top')
    stru.strip('@OW,HW1,HW2')
    u = mda.Universe(stru, f'../gxg_comps/3_res/{label}/traj_2-nw.dcd')

    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)

    # ── Clustering to find dense cluster ──────────────────────────────────────
    ts = u.trajectory[-star]
    centroid, to_cluster, clustered_list, res_list = clustering(u, ts, eps=eps_clust)

    # Build selection strings
    selection = 'resindex ' + ' or resindex '.join(str(r) for r in res_list)

    # ── FIXED: dilute phase = everything NOT in the dense cluster ─────────────
    sel_dilute = np.unique(
        u.select_atoms(f"(nucleic or protein) and not ({selection})").resids
    )
    selection_dilute = ' '.join(str(r) for r in sel_dilute)

    # ── FIXED: both idp_1 and idp_2 are dilute-phase CA only ─────────────────
    idp_1 = u.select_atoms(f"name CA and (resid {selection_dilute})")
    idp_2 = u.select_atoms(f"name CA and (resid {selection_dilute})")

    # ── Intra-chain mask ───────────────────────────────────────────────────────
    resids_1   = idp_1.resids
    resids_2   = idp_2.resids
    chain_1    = resids_1 // chain_size
    chain_2    = resids_2 // chain_size
    intra_mask = (chain_1[:, None] == chain_2[None, :])

    print(f"  idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
          f"chains: {chain_1.max() + 1}  |  dilute residues: {len(sel_dilute)}")

    # ── Contact matrix accumulation ───────────────────────────────────────────
    contMt = []
    for _ in u.trajectory[-star:]:
        contMat = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
        contMat[intra_mask] = False
        contMt.append(sparse.csr_matrix(contMat))
    print(f"  Contact matrix done. ({len(contMt)} frames)")

    # ── Time axis (in ns) ─────────────────────────────────────────────────────
    tM = dt * np.arange(m_corr)   # ns

    # ── Time correlation ───────────────────────────────────────────────────────
    corrN, corr, baseline = timeCorr_boolCsr_fromStart(contMt, len(contMt), m_corr)
    print(f"  Time correlation done.  baseline (frozen fraction) = {baseline/corr[0]:.3f}")

    # ── Exponential fitting ────────────────────────────────────────────────────
    # FIXED: initial guess a=0.5, not 0
    corrFit, para = fit_corr(corrN, tM, [0.5, tM[-1] / 4, 10])
    t1    = para[1]
    t2    = para[1] * para[2]
    t_eff = para[0] * para[1] + (1 - para[0]) * t2
    print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")

    # ── Store results ──────────────────────────────────────────────────────────
    results[label] = {
        'tM':      tM,
        'corrN':   corrN,
        'corr':    corr,
        'baseline': baseline,
        'corrFit': corrFit,
        'para':    para,
        't1':      t1,
        't2':      t2,
        't_eff':   t_eff,
    }

# ── Save all results ───────────────────────────────────────────────────────────
save_path = os.path.join(save_dir, 'corr_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Results saved to {save_path}")


# ══════════════════════════════════════════════════════════════════════════════
#  Load and plot
# ══════════════════════════════════════════════════════════════════════════════

with open(save_path, 'rb') as f:
    results = pickle.load(f)
print(f"  Results loaded from {save_path}")

plt.rcParams.update({"font.size": 18})

# ── Individual plots ───────────────────────────────────────────────────────────
for idx, label in enumerate(systems):
    r = results[label]
    plt.figure()
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],
             label='data')
    plt.plot(r['tM'], r['corrFit'], '--', color=clrs2[idx],
             label=f"fit  τ_eff={r['t_eff']:.2f} ns")
    plt.title(f"Protein contacts — {label}")
    plt.xlabel("t [ns]")
    plt.ylabel("C(t)")
    plt.legend(frameon=False, fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"corr_{label}.png"), dpi=150)
    plt.show()
    print(f"  Saved plot: corr_{label}.png")

# ── Combined comparison plot ───────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
for idx, label in enumerate(systems):
    r = results[label]
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],  alpha=0.4)
    plt.plot(r['tM'], r['corrFit'], '-', color=clrs[idx],
             label=f"{label}  τ_eff={r['t_eff']:.1f} ns")
plt.xlabel("t [ns]")
plt.ylabel("C(t)")
plt.title("Protein contact correlation — all systems")
plt.legend(frameon=False, fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "corr_all_systems.png"), dpi=150)
plt.show()
print("  Saved combined plot: corr_all_systems.png")

# ── τ_eff summary bar chart ────────────────────────────────────────────────────
t_effs = [results[s]['t_eff'] for s in systems]
plt.figure(figsize=(8, 5))
plt.bar(systems, t_effs, color=clrs)
plt.ylabel("τ_eff [ns]")
plt.title("Effective binding lifetime")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "tau_eff_comparison.png"), dpi=150)
plt.show()
print("  Saved τ_eff bar chart: tau_eff_comparison.png")

In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Main analysis loop
# ══════════════════════════════════════════════════════════════════════════════
 
systems    = ['equal', 'aromatic', 'nonpolar', 'polar', 'charged']
clrs       = ['#E01010', '#1010E0', '#10A010', '#A010A0', '#10A0A0']
clrs2      = ['#E09010', '#1090E0', '#10C040', '#C040C0', '#40C0C0']
 
# ── Parameters ────────────────────────────────────────────────────────────────
d_cont     = 8        # Å  contact cutoff
chain_size = 3        # residues per RNA chain
star       = 2000     # frames to analyse
m_corr     = 2000   # correlation window  (~2830)
dt         = 2e-3     # ps per frame → ns  (2 ps = 0.002 ns)
eps_clust  = 9        # DBSCAN eps for clustering
 
save_dir   = './corr_results'
os.makedirs(save_dir, exist_ok=True)
 
results = {}
 
for idx, label in enumerate(systems):
    print(f"\n{'='*60}")
    print(f"  Processing: {label}")
    print(f"{'='*60}")
 
    # ── Load universe ─────────────────────────────────────────────────────────
    stru = parmed.load_file(f'../gxg_comps/9_res/{label}/output.top')
    stru.strip('@OW,HW1,HW2')
    u = mda.Universe(stru, f'../gxg_comps/9_res/{label}/traj-nw.dcd')
 
    a        = u.atoms
    workflow = [trans.wrap(a), trans.unwrap(a)]
    u.trajectory.add_transformations(*workflow)
 
    # ── Clustering to find dense cluster ──────────────────────────────────────
    ts = u.trajectory[-star]
    centroid, to_cluster, clustered_list, res_list = clustering(u, ts, eps=eps_clust)
    
    
    selection = 'resindex '
    for item in res_list:
            selection += str(item)+ ' or resindex '
    selection = selection[0:-13]

    res_list = np.array(res_list)

    selection_2 = ' '
    for item in res_list:
            selection_2 += str(item)+ ' '       

    sel = np.unique(univ.select_atoms(f"(nucleic or protein) and not ({selection})").resids)

    selection_3 = ' '
    for item in sel:
            selection_3 += str(item)+ ' '



    # ── FIXED: both idp_1 and idp_2 are dilute-phase CA only ─────────────────
    idp_1 = u.select_atoms(f"name CA and (resid {selection_2})")[::2]
    idp_2 = u.select_atoms(f"(name CA)")
 
    # ── Intra-chain mask ───────────────────────────────────────────────────────
    resids_1   = idp_1.resids
    resids_2   = idp_2.resids
    chain_1    = resids_1 // chain_size
    chain_2    = resids_2 // chain_size
    intra_mask = (chain_1[:, None] == chain_2[None, :])
 
    print(f"  idp_1: {len(idp_1)} atoms  |  idp_2: {len(idp_2)} atoms  |  "
          f"chains: {chain_1.max() + 1}  |  dilute residues: {len(sel_dilute)}")
 
    # ── Contact matrix accumulation ───────────────────────────────────────────
    contMt = []
    for _ in u.trajectory[-star:]:
        contMat = calc_contact(idp_1, idp_2, d_cont, u.dimensions)
        contMat[intra_mask] = False
        contMt.append(sparse.csr_matrix(contMat))
    print(f"  Contact matrix done. ({len(contMt)} frames)")
 
    # ── Time axis (in ns) ─────────────────────────────────────────────────────
    tM = dt * np.arange(m_corr)   # ns
 
    # ── Time correlation ───────────────────────────────────────────────────────
    corrN, corr = timeCorr_boolCsr_fromStart(contMt, len(contMt), m_corr)
    #print(f"  Time correlation done.  baseline (frozen fraction) = {baseline/corr[0]:.3f}")
 
    # ── Exponential fitting ────────────────────────────────────────────────────
    # FIXED: initial guess a=0.5, not 0
    corrFit, para = fit_corr(corrN, tM, [0.5, tM[-1] / 4, 10])
    t1    = para[1]
    t2    = para[1] * para[2]
    t_eff = para[0] * para[1] + (1 - para[0]) * t2
    print(f"  τ1 = {t1:.3f} ns,  τ2 = {t2:.3f} ns,  τ_eff = {t_eff:.3f} ns")
 
    # ── Store results ──────────────────────────────────────────────────────────
    results[label] = {
        'tM':      tM,
        'corrN':   corrN,
        'corr':    corr,
        #'baseline': baseline,
        'corrFit': corrFit,
        'para':    para,
        't1':      t1,
        't2':      t2,
        't_eff':   t_eff,
    }

#### ── Save all results ───────────────────────────────────────────────────────────
save_path = os.path.join(save_dir, '9_res_corr_results.pkl')
with open(save_path, 'wb') as f:
    pickle.dump(results, f)
print(f"\n  Results saved to {save_path}")


In [ ]:
# ══════════════════════════════════════════════════════════════════════════════
#  Load and plot
# ══════════════════════════════════════════════════════════════════════════════
 
with open(save_path, 'rb') as f:
    results = pickle.load(f)
print(f"  Results loaded from {save_path}")
 
plt.rcParams.update({"font.size": 18})
 
# ── Individual plots ───────────────────────────────────────────────────────────
for idx, label in enumerate(systems):
    r = results[label]
    plt.figure()
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],
             label='data')
    plt.plot(r['tM'], r['corrFit'], '--', color=clrs2[idx],
             label=f"fit  τ_eff={r['t_eff']:.2f} ns")
    plt.title(f"Protein contacts — {label}")
    plt.xlabel("t [ns]")
    plt.ylabel("C(t)")
    plt.legend(frameon=False, fontsize=14)
    plt.tight_layout()
    plt.savefig(os.path.join(save_dir, f"corr_{label}.png"), dpi=150)
    plt.show()
    print(f"  Saved plot: corr_{label}.png")
 
# ── Combined comparison plot ───────────────────────────────────────────────────
plt.figure(figsize=(10, 6))
for idx, label in enumerate(systems):
    r = results[label]
    plt.plot(r['tM'], r['corrN'],   '.', color=clrs[idx],  alpha=0.4)
    plt.plot(r['tM'], r['corrFit'], '-', color=clrs[idx],
             label=f"{label}  τ_eff={r['t_eff']:.1f} ns")
plt.xlabel("t [ns]")
plt.ylabel("C(t)")
plt.title("Protein contact correlation — all systems")
plt.legend(frameon=False, fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "corr_all_systems.png"), dpi=150)
plt.show()
print("  Saved combined plot: corr_all_systems.png")
 
# ── τ_eff summary bar chart ────────────────────────────────────────────────────
t_effs = [results[s]['t_eff'] for s in systems]
plt.figure(figsize=(8, 5))
plt.bar(systems, t_effs, color=clrs)
plt.ylabel("τ_eff [ns]")
plt.title("Effective binding lifetime")
plt.tight_layout()
plt.savefig(os.path.join(save_dir, "tau_eff_comparison.png"), dpi=150)
plt.show()
print("  Saved τ_eff bar chart: tau_eff_comparison.png")

In [ ]:
visul5(u,selection_2,selection_3)

In [ ]:
u.select_atoms('protein').resids

In [ ]:
# Check how many contacts persist across ALL frames
persistent = contMt[0].toarray()
for mat in contMt[1:]:
    persistent = persistent & mat.toarray()

frozen_fraction = persistent.sum() / contMt[0].sum()
print(f"Frozen fraction: {frozen_fraction:.3f}")   # expect ~0.8

In [ ]:
persistent